<a href="https://colab.research.google.com/github/deacs11/Intelligent-Documentation-Builder/blob/main/Intelligent_Documentation_Builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup e Configuration

In [2]:
# 🔧 Librerie e chiavi API
!pip install -q pyautogen openai python-dotenv fpdf2

from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
from google.colab import userdata
import os

# 🔐 Carica la chiave API da Colab Secrets
api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("❌ Nessuna API key trovata! Controlla i Secrets.")
print("✅ API caricata correttamente.")

# 💡 Config LLM
llm_config = {
    "config_list": [{
        "model": "gpt-4",
        "api_key": api_key
    }]
}


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.7/781.7 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.7/245.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.9 MB/s eta 0:00:00
✅ API caricata correttamente.


Define the Documentation Agents

In [3]:
technical_writer = AssistantAgent(
    name="TechnicalWriter",
    system_message="You are a technical writer. You convert complex content into clear, professional documentation.",
    llm_config=llm_config
)

language_editor = AssistantAgent(
    name="LanguageEditor",
    system_message="You are a language editor. You correct grammar, spelling, and improve clarity.",
    llm_config=llm_config
)

logic_checker = AssistantAgent(
    name="LogicChecker",
    system_message="You verify the logical coherence of the content. Point out inconsistencies or vague sections.",
    llm_config=llm_config
)

semantic_reviewer = AssistantAgent(
    name="SemanticReviewer",
    system_message="You ensure that terms and technical vocabulary are used consistently and appropriately.",
    llm_config=llm_config
)

formality_adjuster = AssistantAgent(
    name="FormalityAdjuster",
    system_message="You review tone and formality according to the desired audience (e.g., internal, public, academic).",
    llm_config=llm_config
)

User Input Agent

In [4]:
user_proxy = UserProxyAgent(
    name="HumanModerator",
    human_input_mode="NEVER",
    code_execution_config={"work_dir": ".", "use_docker": False}
)


GroupChat Setup

In [5]:
agents = [
    technical_writer,
    language_editor,
    logic_checker,
    semantic_reviewer,
    formality_adjuster,
    user_proxy
]

documentation_chat = GroupChat(
    agents=agents,
    messages=[],
    max_round=5
)

chat_manager = GroupChatManager(
    groupchat=documentation_chat,
    llm_config=llm_config
)

Launch the Collaboration

In [18]:
input_script = """
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] < right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result
"""


user_proxy.initiate_chat(
    chat_manager,
    message=f"""
I need you all to work together to produce a clean, clear, and professionally written documentation for the following code:

{input_script}
"""
)

HumanModerator (to chat_manager):


I need you all to work together to produce a clean, clear, and professionally written documentation for the following code:


def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] < right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result



--------------------------------------------------------------------------------

Next speaker: TechnicalWriter

TechnicalWriter (to chat_manager):

**Merge Sort Function Documentation**

---
*Merge_sort is a recursive function that sorts an array of items in ascending order using the divide and conquer strategy. Here's ho

ChatResult(chat_id=None, chat_history=[{'content': '\nI need you all to work together to produce a clean, clear, and professionally written documentation for the following code:\n\n\ndef merge_sort(arr):\n    if len(arr) <= 1:\n        return arr\n    mid = len(arr) // 2\n    left = merge_sort(arr[:mid])\n    right = merge_sort(arr[mid:])\n    return merge(left, right)\n\ndef merge(left, right):\n    result = []\n    i = j = 0\n    while i < len(left) and j < len(right):\n        if left[i] < right[j]:\n            result.append(left[i])\n            i += 1\n        else:\n            result.append(right[j])\n            j += 1\n    result.extend(left[i:])\n    result.extend(right[j:])\n    return result\n\n', 'role': 'assistant', 'name': 'HumanModerator'}, {'content': "**Merge Sort Function Documentation**\n\n---\n*Merge_sort is a recursive function that sorts an array of items in ascending order using the divide and conquer strategy. Here's how it works:*\n\n**Function Prototype**\n`

Extract and Summarize

In [19]:
synthesizer = AssistantAgent(
    name="Synthesizer",
    system_message="Summarize the following discussion into a well-structured documentation summary.",
    llm_config=llm_config
)

conversation_text = "\n\n".join([
    f"{msg['role']}: {msg['content']}" for msg in documentation_chat.messages
])

response = synthesizer.generate_reply(messages=[
    {"role": "user", "content": f"Here is the full conversation:\n\n{conversation_text}\n\nPlease summarize it."}
])

summary = response

In [20]:
pip install fpdf

In [21]:
import unicodedata

# 🔧 Remove all characters not supported by 'latin-1'
def normalize_to_ascii(text):
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')

# ✂️ Normalize the transcript and summary
clean_transcript = normalize_to_ascii(conversation_text)
clean_summary = normalize_to_ascii(summary)

Save Output as PDF

In [22]:
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()

pdf.set_font("Helvetica", "B", size=14)
pdf.cell(200, 10, "Documentation Generation Output", ln=True, align="C")

pdf.ln(10)
pdf.set_font("Helvetica", "B", size=12)
pdf.cell(200, 10, "Debate Transcript", ln=True)

pdf.set_font("Helvetica", "", size=11)
pdf.multi_cell(0, 8, clean_transcript)

pdf.ln(10)
pdf.set_font("Helvetica", "B", size=12)
pdf.cell(200, 10, "AI-Generated Summary", ln=True)

pdf.set_font("Helvetica", "", size=11)
pdf.multi_cell(0, 8, clean_summary)

pdf.output("generated_documentation.pdf")

/tmp/ipython-input-22-1788072974.py:7: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(200, 10, "Documentation Generation Output", ln=True, align="C")
/tmp/ipython-input-22-1788072974.py:11: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(200, 10, "Debate Transcript", ln=True)
/tmp/ipython-input-22-1788072974.py:18: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(200, 10, "AI-Generated Summary", ln=True)
